In [1]:
import torch


In [172]:
n_embed=32
dropout=0.2
n_layer=6
n_head=4
batch_size=64
block_size=256
learning_rate=3e-4
n_embed=100
eval_interval=300
eval_iters=200
device='cpu'


In [141]:
with open('input.txt','r',encoding='utf-8') as t:
    input=t.read()

In [142]:
vocab=sorted(list(set(input)))
vocab_size=len(vocab)
char2int={c:i for i,c in enumerate(vocab)}
int2char={i:c for i,c in enumerate(char2int)}
encode=lambda s: [char2int[c] for c in s]
decode=lambda l:''.join([int2char[i] for i in l])

In [143]:
data=torch.tensor(encode(input),dtype=torch.long)

In [144]:
n=int(0.9*len(data))
train_data=data[:n]
val_data=train_data[n:]

In [145]:
block_size=8
x=train_data[:block_size]
y=train_data[1:block_size+1]
for i in range(block_size):
    context=x[:i+1]
    target=y[i]
    print(f"when context is {context}, target is {target}")

when context is tensor([18]), target is 47
when context is tensor([18, 47]), target is 56
when context is tensor([18, 47, 56]), target is 57
when context is tensor([18, 47, 56, 57]), target is 58
when context is tensor([18, 47, 56, 57, 58]), target is 1
when context is tensor([18, 47, 56, 57, 58,  1]), target is 15
when context is tensor([18, 47, 56, 57, 58,  1, 15]), target is 47
when context is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target is 58


In [146]:
batch_size=4
ix=torch.randint(len(train_data)-block_size,(batch_size,))
x=torch.stack([train_data[i:i+block_size] for i in ix])
y=torch.stack([train_data[i+1:i+1+block_size] for i in ix])
x

tensor([[ 1, 45, 43, 52, 58, 50, 43,  6],
        [53, 50, 53, 59, 56, 43, 42, 10],
        [ 6,  1, 39, 52, 42,  1, 61, 47],
        [26, 15, 17, 10,  0, 26, 53, 58]])

In [162]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [163]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [147]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.k=nn.Linear(n_embed,head_size,bias=False)
        self.q=nn.Linear(n_embed,head_size,bias=False)
        self.v=nn.Linear(n_embed,head_size,bias=False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        B,T,C=x.shape
        K=self.k(x)
        Q=self.q(x)
        V=self.v(x)
        wei=Q@K.transpose(-2,-1)*C**-0.5
        tril=torch.tril((torch.ones(block_size,block_size)))
        #wei=torch.zeros((block_size,block_size))
        wei=wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))
        wei=F.softmax(wei,dim=1)
        wei=self.dropout(wei)
        out=wei@V
        return(out)

In [148]:
class MultiHead(nn.Module):
    def __init__(self,num_heads,head_size):
        super().__init__()
        self.heads=nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj=nn.Linear(n_embed,n_embed)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        out=torch.cat([h(x) for h in self.heads],dim=-1)
        out=self.dropout(self.proj(out))
        return(out)

In [149]:
class FeedForward(nn.Module):
    def __init__(self,n_embed):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(n_embed,4*n_embed),
            nn.ReLU(),
            nn.Linear(4*n_embed,n_embed),
            nn.Dropout(dropout),
        )
    def forward(self,x):
        return self.net(x)


In [150]:
class Block(nn.Module):
    def __init__(self,n_embed,n_head):
        super().__init__()
        head_size=n_embed//n_head
        self.sa=MultiHead(n_head,head_size)
        self.ffwd=FeedForward(n_embed)
        self.ln1=nn.LayerNorm(n_embed)
        self.ln2=nn.LayerNorm(n_embed)
    def forward(self,x):
        x=x+self.sa(self.ln1(x))
        x=x+self.ffwd(self.ln2(x))
        return x


In [158]:
import torch.nn as nn
from torch.nn import functional as F 
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,n_embed)
        self.position_embedding_table=nn.Embedding(block_size,n_embed)
        self.blocks=nn.Sequential(*[Block(n_embed,n_head=n_head) for _ in range(n_layer)])
        self.ln_f=nn.LayerNorm(n_embed)
        self.lm_head=nn.Linear(n_embed,vocab_size)
    def forward(self,idx,targets=None):
        B,T=idx.shape
        tok_embed=self.token_embedding_table(idx)
        pos_embed=self.position_embedding_table(torch.arange(T,device='cpu'))
        x=tok_embed+pos_embed
        x=self.blocks(x)
        x=self.ln_f(x)
        logits=self.lm_head(x)
        if targets==None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits,targets)
        return(logits,loss)
    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond=idx[:,-block_size:]
            logits,loss=self(idx_cond)
            logits=logits[:, -1, :]
            probs=F.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return(idx)

In [168]:
model=BigramLanguageModel()

#logits,loss=m(x,y)

In [160]:
#batch_size=32
optimizer=torch.optim.AdamW(m.parameters(),lr=learning_rate)
for i in range(2000):
    ix=torch.randint(len(train_data)-block_size,(batch_size,))
    x=torch.stack([train_data[i:i+block_size] for i in ix])
    y=torch.stack([train_data[i+1:i+1+block_size] for i in ix])
    logits,loss=m(x,y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())


KeyboardInterrupt: 

In [173]:
max_iters=500
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



RuntimeError: random_ expects 'from' to be less than 'to', but got from=0 >= to=-256

In [154]:
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long),max_new_tokens=1000)[0].tolist()))


EMM.
MnAEENDLu
'LQO yIh! SVyI ngoCSe aI avae r!or thb baYt'm aO
AFrep, Jekt crge
NanhB
UhE xd F,
D faSut bout;
Uat heamke hache.
uosy beguesiQrt amere EouBlomu. seecor dortbei.
:ldoS
A:hok, athem te,gWo,
O boy'fnd c,coesy nwgeate yareeafd ve
Com ha otkam e uraw nafriy lafame?

WhiD swiDr chaveebeen't ytusos. bhe' yleith mar,te
 OA;
Ik, horallon yof,,
Th
heglle, rofw, wre
Ak.
Te phanoigo fad?
t! y3hWhacla, asa? tholont youlle?t chischals lmryn't
DTUX.
HRDAE ilt wapre ay Ian I atlllecond:
eae
Heasif mt
avefSI asihe whe'e PtH
I Ior mey priO chiI ftear:om anhet uormoy harekiwn.
I yofsid winllh kar avo ammeasugrdeRF?u
''ut, kodird base y-Re acrohu
wert maELSho.
h apusot cakerloRs
DAE ruralysoniu, apubt aut mIIN Hamed bQgas,
I
igk woror anutle,
d som yEu itC;:
Dd,K
jK
UDfAAllve
F!
Fh ahhat wisiwos icce hUiE bI$llsigrin ousad Hant heere aJee, yuse, ifRonlsong
hUI kEVurF cintinte
O 'wcamIUHFe am
Ylyoly oEyousd amXe atrime? yt
KYhu loucom oweo lmd hold imem
Ferithe! Dbyves,

zom obonot, behes 